# OCSF placeholder tokenization (ver_1)

Format-preserving encryption of `[PREFIX_SUFFIX]` placeholders in OCSF logs,
with a vault of issued tokens so a detokenization request can be validated
before any key is used.

The cell tagged **`library`** below holds the whole module. `test_encryp.ipynb`
executes exactly that cell, so keep new scratch work in a separate, untagged
cell.

## Environment

| Variable | Required | Purpose |
| --- | --- | --- |
| `FF3_KEY` / `FF3_TWEAK` | yes (or the versioned form) | A single key, named by `FF3_KEY_VERSION` |
| `FF3_KEY_VERSION` | no (`v1`) | Name of the unversioned key |
| `FF3_KEY_<V>` / `FF3_TWEAK_<V>` | for a key ring | One pair per version, e.g. `FF3_KEY_V2` |
| `FF3_ACTIVE_KEY_VERSION` | no | Which version encrypts new tokens |
| `DATABASE_URL` | for DB use | psycopg connection string |

`pip install ff3 psycopg[binary] pytest` — `psycopg` is imported lazily, so
everything except `connect_database()` works without it installed.

## Usage

```python
conn = connect_database()
ensure_schema(conn)
key_ring = load_key_ring()

# Ingest: one transaction per log, encrypted with the ring's active key.
safe_log = tokenize_log(raw_log, key_ring, conn, case_id="INC-001")

# Analyst hands a token back; unissued tokens are refused and audited.
original = safe_decrypt(conn, key_ring, "[HOST_V6PL]", "INC-001", actor="analyst")

# Or reverse a whole document at once.
restored = detokenize_log(safe_log, conn, key_ring, "INC-001", actor="analyst")
```

## Key rotation

Every token records the key version that encrypted it, and the ring keeps
every version that can still decrypt. Detokenization resolves the key from
the token rather than assuming the active one:

```
vault_schema.issued_tokens
    token       "[HOST_V6PL]"
    key_version "v1"
          |
          v
    KeyRing.for_version("v1")
          |
    +-----+-----+
    | v1  | v2  |  <- active
    +-----+-----+
          |
          v
    decrypt with v1   -> "[HOST_01]"
```

So rotating is additive, and old tokens keep working:

```bash
export FF3_KEY_V2=... FF3_TWEAK_V2=...   # add the new key
export FF3_ACTIVE_KEY_VERSION=v2         # new tokens use it; v1 still decrypts
```

`key_versions_in_use(conn)` counts tokens per version, so a key comes off the
ring when its count is zero — not on a guess. Taking a key off the ring while
tokens still reference it raises `UnknownKeyVersionError`, which is the one
genuine "cannot decrypt" case; an *older but loaded* version is not an error.

`KeyRing` is the provider seam. `load_key_ring()` fills it from environment
variables; a KMS or Vault client can build the same `{version: VersionedCipher}`
mapping and hand it to `KeyRing(...)` directly.

Rotation does not re-encrypt what already exists. Moving old tokens onto the
new key means re-issuing them and rewriting the stored logs that contain them
— a data migration, deliberately outside this module.

## Invariants worth keeping

- **The vault owns the prefix.** Ciphertext may contain `_`, so a token cannot
  be re-split with a regex; `lookup_issued_token` returns the prefix and the
  original suffix length that were recorded at issuance.
- **The vault owns the key version too.** `record_issued_token` takes it from
  the `VersionedCipher` that did the encryption, never from the environment,
  so a token cannot be stamped with a version that did not encrypt it.
- **Suffixes are alphanumeric only** (`PLACEHOLDER_RE`). That is what makes
  `_` usable as padding without two different values encrypting to one token.
- **Nothing below `tokenize_log` / `safe_decrypt` / `detokenize_log` commits.**
  Those three own the transaction boundary.

## Known limits

- Letter case is not preserved: `[host_ab]` round-trips to `[HOST_AB]`.
- Tokens are deterministic across cases — the same host yields the same token
  in every case, so tokens can be correlated between cases. Derive the tweak
  from `case_id` if that is not wanted.
- Small suffix domains are enumerable: `[HOST_NN]` has 100 values, so anyone
  who can call `tokenize_field` can build a full lookup table without the key.
- `actor` is recorded, not authenticated.

In [ ]:
from __future__ import annotations

import os
import re
from typing import TYPE_CHECKING, Any, Iterable, Mapping

from ff3 import FF3Cipher

if TYPE_CHECKING:  # psycopg is only needed to talk to a real database
    import psycopg

# The pad character must be part of the cipher alphabet but must never appear
# in a real suffix, otherwise padding is not reversible: rjust("01", 4, "_")
# and the literal suffix "__01" would encrypt to the very same token.
PAD_CHAR = "_"
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789" + PAD_CHAR

# Suffixes are alphanumeric only, so they can never collide with padding.
PLACEHOLDER_RE = re.compile(r"\[([A-Za-z]+(?:_[A-Za-z]+)*)_([A-Za-z0-9]+)\]")

# Ciphertext may legitimately contain PAD_CHAR, so a token is not always
# shaped like a placeholder. Detokenization scans this broader pattern and
# lets the vault decide what is really a token.
TOKEN_CANDIDATE_RE = re.compile(r"\[[A-Za-z0-9_]+\]")

DEFAULT_KEY_VERSION = "v1"
KEY_ENV_RE = re.compile(r"^FF3_KEY_([A-Za-z0-9]+)$")
RESERVED_KEY_ENV = {"FF3_KEY_VERSION"}


class TokenHallucinationError(Exception):
    """Raised when a token was never issued for the case that asks for it."""

    def __init__(self, token: str, case_id: str) -> None:
        self.token = token
        self.case_id = case_id
        super().__init__(f"Token was not issued for case_id={case_id!r}: {token!r}")


class TokenLengthError(ValueError):
    """Raised when a suffix does not fit the cipher's length bounds."""


class TokenCollisionError(RuntimeError):
    """Raised when one token string maps to two different plaintexts."""

    def __init__(self, token: str, case_id: str) -> None:
        self.token = token
        self.case_id = case_id
        super().__init__(
            f"Token {token!r} is already issued for case_id={case_id!r} with "
            "different parameters; it cannot be decrypted unambiguously"
        )


class UnknownKeyVersionError(RuntimeError):
    """Raised when the key that issued a token is not loaded in the key ring.

    This is the genuine "cannot decrypt" case. A token issued under a key
    version that is merely *older* than the active one decrypts normally, as
    long as that version is still on the ring.
    """

    def __init__(self, version: str, available: tuple[str, ...], token: str | None = None) -> None:
        self.version = version
        self.available = tuple(available)
        self.token = token
        super().__init__(
            f"key version {version!r} is not on the key ring "
            f"(loaded: {', '.join(available) or 'none'}); load that key to "
            "detokenize tokens issued under it"
        )


class VersionedCipher:
    """An FF3 cipher bound to the key version that names it.

    Carrying the two together is what stops a token from being stamped with a
    version that does not describe the key that actually encrypted it.
    """

    __slots__ = ("version", "cipher")

    def __init__(self, version: str, cipher: FF3Cipher) -> None:
        self.version = version.lower()
        self.cipher = cipher

    @property
    def minLen(self) -> int:
        return self.cipher.minLen

    @property
    def maxLen(self) -> int:
        return self.cipher.maxLen

    def encrypt(self, plaintext: str) -> str:
        return self.cipher.encrypt(plaintext)

    def decrypt(self, ciphertext: str) -> str:
        return self.cipher.decrypt(ciphertext)

    def __repr__(self) -> str:  # never render key material
        return f"VersionedCipher(version={self.version!r})"


class KeyRing:
    """Every key version that can still decrypt, plus the one that encrypts.

    Rotation adds a version and moves `active_version` forward; the older
    versions stay on the ring, so tokens issued under them keep decrypting:

        token.key_version -> KeyRing.for_version() -> that version's cipher

    This is the key provider seam. `load_key_ring()` fills it from the
    environment, but a KMS or Vault client can build the same mapping.
    """

    def __init__(self, ciphers: Iterable[VersionedCipher], active_version: str) -> None:
        # Indexed by each cipher's own version, so the name a token is stamped
        # with and the name the ring is searched by cannot drift apart.
        self._ciphers = {cipher.version: cipher for cipher in ciphers}
        active = active_version.lower()
        if active not in self._ciphers:
            raise RuntimeError(
                f"active key version {active_version!r} is not on the key ring "
                f"(loaded: {', '.join(sorted(self._ciphers)) or 'none'})"
            )
        self.active_version = active

    @property
    def active(self) -> VersionedCipher:
        """The cipher that encrypts new tokens."""
        return self._ciphers[self.active_version]

    @property
    def versions(self) -> tuple[str, ...]:
        """Every version that can still be decrypted."""
        return tuple(sorted(self._ciphers))

    def for_version(self, version: str) -> VersionedCipher:
        """Resolve the cipher a token was issued under."""
        try:
            return self._ciphers[version.lower()]
        except KeyError:
            raise UnknownKeyVersionError(version, self.versions) from None

    def __repr__(self) -> str:
        return f"KeyRing(versions={self.versions}, active={self.active_version!r})"


def _key_material(environ: Mapping[str, str]) -> dict[str, tuple[str, str | None]]:
    """Collect {version: (key, tweak)} pairs out of the environment."""
    material: dict[str, tuple[str, str | None]] = {}

    # Unversioned pair, for single-key deployments.
    if environ.get("FF3_KEY"):
        version = (environ.get("FF3_KEY_VERSION") or DEFAULT_KEY_VERSION).lower()
        material[version] = (environ["FF3_KEY"], environ.get("FF3_TWEAK"))

    # Versioned pairs win over the unversioned one for the same version.
    for name, key in environ.items():
        if name in RESERVED_KEY_ENV or not key:
            continue
        match = KEY_ENV_RE.match(name)
        if match is None:
            continue
        suffix = match.group(1)
        material[suffix.lower()] = (key, environ.get(f"FF3_TWEAK_{suffix}"))

    return material


def load_key_ring(environ: Mapping[str, str] | None = None) -> KeyRing:
    """Build the key ring from environment variables.

    | Variable | Meaning |
    | --- | --- |
    | `FF3_KEY` / `FF3_TWEAK` | a single key, named by `FF3_KEY_VERSION` (default `v1`) |
    | `FF3_KEY_<V>` / `FF3_TWEAK_<V>` | one pair per version, e.g. `FF3_KEY_V2` |
    | `FF3_ACTIVE_KEY_VERSION` | which version encrypts new tokens |

    To rotate: add the new pair, point `FF3_ACTIVE_KEY_VERSION` at it, and
    leave the old pair loaded until `key_versions_in_use()` reports nothing
    left under it. Version names are matched case-insensitively.
    """
    environ = os.environ if environ is None else environ
    material = _key_material(environ)
    if not material:
        raise RuntimeError("FF3_KEY (or FF3_KEY_<VERSION>) environment variable is required")

    ciphers = []
    for version, (key, tweak) in material.items():
        if not tweak:
            raise RuntimeError(f"FF3_TWEAK for key version {version!r} is required")
        ciphers.append(
            VersionedCipher(version, FF3Cipher.withCustomAlphabet(key, tweak, ALPHABET))
        )

    requested = environ.get("FF3_ACTIVE_KEY_VERSION") or environ.get("FF3_KEY_VERSION")
    if requested:
        active = requested
    elif len(ciphers) == 1:
        active = ciphers[0].version
    else:
        active = DEFAULT_KEY_VERSION
    return KeyRing(ciphers, active)


def connect_database() -> "psycopg.Connection":
    database_url = os.environ.get("DATABASE_URL")
    if not database_url:
        raise RuntimeError("DATABASE_URL environment variable is required")
    import psycopg

    return psycopg.connect(database_url)


ISSUED_TOKENS_SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS vault_schema;

CREATE TABLE IF NOT EXISTS vault_schema.issued_tokens (
    id          BIGSERIAL PRIMARY KEY,
    token       TEXT NOT NULL,
    prefix      TEXT NOT NULL,
    suffix_len  INTEGER NOT NULL,
    case_id     TEXT NOT NULL,
    key_version TEXT NOT NULL,
    issued_at   TIMESTAMPTZ NOT NULL DEFAULT now(),
    UNIQUE (token, case_id)
);
"""

DETOKENIZE_LOG_SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS metadata_schema;

CREATE TABLE IF NOT EXISTS metadata_schema.detokenize_log (
    id             BIGSERIAL PRIMARY KEY,
    token          TEXT NOT NULL,
    case_id        TEXT NOT NULL,
    actor          TEXT NOT NULL,
    outcome        TEXT NOT NULL CHECK (outcome IN ('success', 'rejected')),
    detokenized_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS detokenize_log_case_idx
    ON metadata_schema.detokenize_log (case_id, detokenized_at DESC);
"""

SCHEMA_SQL = ISSUED_TOKENS_SCHEMA_SQL + DETOKENIZE_LOG_SCHEMA_SQL


def ensure_schema(conn: "psycopg.Connection") -> None:
    """Create both schemas if they are missing; safe to run repeatedly.

    Executed without parameters on purpose: psycopg only allows several
    statements in one execute() over the simple query protocol, which it uses
    when no parameters are passed.
    """
    with conn.cursor() as cursor:
        cursor.execute(SCHEMA_SQL)
    conn.commit()


def lookup_issued_token(
    conn: "psycopg.Connection", token: str, case_id: str
) -> tuple[str, int, str] | None:
    """Return (prefix, suffix_len, key_version) for an issued token, or None.

    The prefix is read back from the vault instead of being re-parsed out of
    the token, because ciphertext can contain PAD_CHAR and would otherwise be
    split in a different place than it was at issuance time.
    """
    with conn.cursor() as cursor:
        cursor.execute(
            """
            SELECT prefix, suffix_len, key_version
            FROM vault_schema.issued_tokens
            WHERE token = %s AND case_id = %s
            """,
            (token, case_id),
        )
        row = cursor.fetchone()
    return None if row is None else (row[0], int(row[1]), row[2])


def is_token_issued(conn: "psycopg.Connection", token: str, case_id: str) -> bool:
    """Return whether this exact token was issued for this case."""
    return lookup_issued_token(conn, token, case_id) is not None


def key_versions_in_use(
    conn: "psycopg.Connection", case_id: str | None = None
) -> dict[str, int]:
    """Count issued tokens per key version.

    A key can only be taken off the ring once its count here reaches zero,
    so this is what makes retiring a rotated-out key a decision rather than a
    guess.
    """
    with conn.cursor() as cursor:
        if case_id is None:
            cursor.execute(
                "SELECT key_version, count(*) FROM vault_schema.issued_tokens "
                "GROUP BY key_version"
            )
        else:
            cursor.execute(
                "SELECT key_version, count(*) FROM vault_schema.issued_tokens "
                "WHERE case_id = %s GROUP BY key_version",
                (case_id,),
            )
        return {version: int(count) for version, count in cursor.fetchall()}


def record_issued_token(
    conn: "psycopg.Connection",
    token: str,
    prefix: str,
    suffix_len: int,
    case_id: str,
    key_version: str,
) -> None:
    """Record an issued token without creating duplicate case records.

    `key_version` is required and comes from the VersionedCipher that did the
    encryption, never from the environment, so a token cannot be stamped with
    a version that did not encrypt it.

    Does not commit: the caller owns the transaction so that a whole log is
    tokenized atomically. Use tokenize_log for that.
    """
    with conn.cursor() as cursor:
        cursor.execute(
            """
            INSERT INTO vault_schema.issued_tokens
                (token, prefix, suffix_len, case_id, key_version)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (token, case_id) DO NOTHING
            """,
            (token, prefix, suffix_len, case_id, key_version),
        )
        if cursor.rowcount == 1:
            return

    # The row already existed. Normally that is the identical value seen
    # again, but two different plaintexts can in principle produce the same
    # token string; that is unrecoverable, so fail loudly here instead of
    # decrypting the wrong value later.
    if lookup_issued_token(conn, token, case_id) != (prefix, suffix_len, key_version):
        raise TokenCollisionError(token, case_id)


def record_detokenize_attempt(
    conn: "psycopg.Connection", token: str, case_id: str, actor: str, outcome: str
) -> None:
    """Audit a detokenization attempt, including the ones that were refused."""
    with conn.cursor() as cursor:
        cursor.execute(
            """
            INSERT INTO metadata_schema.detokenize_log
                (token, case_id, actor, outcome, detokenized_at)
            VALUES (%s, %s, %s, %s, now())
            """,
            (token, case_id, actor, outcome),
        )


def _require_issuance_pair(conn: Any, case_id: str | None) -> None:
    if (conn is None) != (case_id is None):
        raise ValueError(
            "conn and case_id must be given together: a token issued without "
            "a vault record can never be detokenized"
        )


def _encrypt_suffix(versioned: VersionedCipher, suffix: str) -> str:
    padded_suffix = suffix.rjust(max(versioned.minLen, len(suffix)), PAD_CHAR)
    if len(padded_suffix) > versioned.maxLen:
        raise TokenLengthError(
            f"suffix of length {len(suffix)} exceeds the cipher maximum of "
            f"{versioned.maxLen} characters"
        )
    return versioned.encrypt(padded_suffix)


def tokenize_field(
    key_ring: KeyRing,
    value: str,
    conn: "psycopg.Connection | None" = None,
    case_id: str | None = None,
) -> str:
    """Encrypt only placeholder suffixes, including placeholders in longer text.

    Always encrypts with the ring's active key and records that version
    alongside the token.

    Suffixes are upper-cased to fit the cipher alphabet, so the original
    letter case is not restored on the way back. Tokenizing an already
    tokenized value encrypts it a second time; run this once per log.

    Does not commit; see tokenize_log.
    """
    _require_issuance_pair(conn, case_id)
    versioned = key_ring.active

    def replace_placeholder(match: re.Match[str]) -> str:
        prefix = match.group(1).upper()
        suffix = match.group(2).upper()
        token = f"[{prefix}_{_encrypt_suffix(versioned, suffix)}]"
        if conn is not None and case_id is not None:
            record_issued_token(
                conn, token, prefix, len(suffix), case_id, versioned.version
            )
        return token

    return PLACEHOLDER_RE.sub(replace_placeholder, value)


def walk_and_tokenize(
    node: Any,
    key_ring: KeyRing,
    conn: "psycopg.Connection | None" = None,
    case_id: str | None = None,
) -> Any:
    """Recursively tokenize placeholders while preserving the surrounding JSON."""
    _require_issuance_pair(conn, case_id)
    if isinstance(node, dict):
        return {
            key: walk_and_tokenize(value, key_ring, conn, case_id)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [walk_and_tokenize(value, key_ring, conn, case_id) for value in node]
    if isinstance(node, str):
        return tokenize_field(key_ring, node, conn, case_id)
    return node


def tokenize_log(
    node: Any, key_ring: KeyRing, conn: "psycopg.Connection", case_id: str
) -> Any:
    """Tokenize a complete log in a single transaction.

    One commit per log rather than one per placeholder: a log that fails
    halfway through leaves no partially issued tokens behind.
    """
    result = walk_and_tokenize(node, key_ring, conn, case_id)
    conn.commit()
    return result


def _detokenize_token(
    conn: "psycopg.Connection", key_ring: KeyRing, token: str, case_id: str, actor: str
) -> str:
    """Validate issuance, resolve the issuing key, decrypt, and audit.

    Does not commit.
    """
    issued = lookup_issued_token(conn, token, case_id)
    if issued is None:
        record_detokenize_attempt(conn, token, case_id, actor, "rejected")
        raise TokenHallucinationError(token, case_id)

    prefix, suffix_len, key_version = issued
    try:
        # The vault says which key issued this token; the ring supplies it.
        # An older version decrypts exactly as well as the active one.
        versioned = key_ring.for_version(key_version)
    except UnknownKeyVersionError as error:
        error.token = token
        record_detokenize_attempt(conn, token, case_id, actor, "rejected")
        raise

    # Token is "[" + prefix + "_" + ciphertext + "]". Slicing by the recorded
    # prefix length makes the split match the one used at issuance time even
    # when the ciphertext itself contains PAD_CHAR.
    ciphertext = token[len(prefix) + 2 : -1]
    original_suffix = versioned.decrypt(ciphertext)[-suffix_len:]
    record_detokenize_attempt(conn, token, case_id, actor, "success")
    return f"[{prefix}_{original_suffix}]"


def safe_decrypt(
    conn: "psycopg.Connection",
    key_ring: KeyRing,
    token: str,
    case_id: str,
    actor: str,
) -> str:
    """Validate issuance before decrypting and audit every attempt.

    `actor` is trusted as given: this layer records who the caller claims to
    be, it does not authenticate them. Authorization belongs upstream.
    """
    try:
        return _detokenize_token(conn, key_ring, token, case_id, actor)
    finally:
        # Commit in finally so a refused attempt is audited too.
        conn.commit()


def detokenize_field(
    value: str,
    conn: "psycopg.Connection",
    key_ring: KeyRing,
    case_id: str,
    actor: str,
) -> str:
    """Replace every issued token inside a string with its original value.

    Bracketed text that was never issued is left alone unless it is shaped
    like a placeholder, which means something invented it. Does not commit.
    """

    def replace_token(match: re.Match[str]) -> str:
        candidate = match.group(0)
        if is_token_issued(conn, candidate, case_id):
            return _detokenize_token(conn, key_ring, candidate, case_id, actor)
        if PLACEHOLDER_RE.fullmatch(candidate) is None:
            return candidate  # e.g. "logs[6124]", not a token at all
        record_detokenize_attempt(conn, candidate, case_id, actor, "rejected")
        raise TokenHallucinationError(candidate, case_id)

    return TOKEN_CANDIDATE_RE.sub(replace_token, value)


def walk_and_detokenize(
    node: Any,
    conn: "psycopg.Connection",
    key_ring: KeyRing,
    case_id: str,
    actor: str,
) -> Any:
    """Recursively detokenize a JSON structure. Does not commit."""
    if isinstance(node, dict):
        return {
            key: walk_and_detokenize(value, conn, key_ring, case_id, actor)
            for key, value in node.items()
        }
    if isinstance(node, list):
        return [walk_and_detokenize(value, conn, key_ring, case_id, actor) for value in node]
    if isinstance(node, str):
        return detokenize_field(node, conn, key_ring, case_id, actor)
    return node


def detokenize_log(
    node: Any,
    conn: "psycopg.Connection",
    key_ring: KeyRing,
    case_id: str,
    actor: str,
) -> Any:
    """Detokenize a complete log, auditing every token it touches.

    Tokens issued under different key versions can appear side by side in one
    log; each is resolved against the version recorded for it.
    """
    try:
        return walk_and_detokenize(node, conn, key_ring, case_id, actor)
    finally:
        conn.commit()